# Laboratorio 21. Sistemas de recomendación con embeddings

**Pregunta guía:** ¿Cómo aprenden la factorización matricial y un modelo de filtrado colaborativo neuronal a recomendar productos a partir de interacciones implícitas, y cómo sabemos si superan a recomendar lo más popular?

## Objetivos
Al finalizar podrás construir datos de interacción implícita con particiones *leave-one-out*; entrenar factorización matricial (MF) y *Neural Collaborative Filtering* (NCF) con muestreo negativo; evaluar con HitRate@10, NDCG@10 y cobertura de catálogo; e interpretar similitud entre productos con sus embeddings.

## Prerrequisitos y conexión
Extiende los embeddings del laboratorio 20: ahora tanto usuarios como productos tienen un vector. Cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
En MF, la afinidad entre el usuario $u$ y el producto $i$ es $\hat{s}_{ui}=p_u^\top q_i+b_u+b_i$, con embeddings $p_u, q_i\in\mathbb{R}^d$ y sesgos. NCF reemplaza el producto punto por una red: $\hat{s}_{ui}=f_\theta([p_u;q_i])$, que puede representar interacciones no lineales. Con datos implícitos (compras, clics) solo observamos positivos, así que en cada época **muestreamos negativos**: productos que el usuario no ha consumido, tratados como etiqueta 0, y optimizamos entropía cruzada binaria.

## Problema y datos
Un retailer ficticio con 500 clientes y 150 productos en 6 categorías quiere recomendar 10 productos por cliente. Generamos preferencias latentes: cada cliente tiene gustos y cada producto pertenece a una categoría con un perfil común, más un efecto de popularidad. Todos los datos se generan en el notebook.

## Perspectiva de negocio y datos
Un recomendador decide qué ve cada cliente. Además de precisión, importa la **cobertura**: si solo recomienda los éxitos de ventas, concentra la demanda y deja parte del catálogo invisible. También importa qué datos de comportamiento se usan y con qué consentimiento.

## Experimento conceptual
¿Superan MF y NCF a la popularidad en HitRate@10, y a qué costo en cobertura? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional. Usamos `rng` (generador de NumPy) para los datos y `torch.manual_seed` antes de cada entrenamiento para que las comparaciones entre modelos sean justas.

In [ ]:
import copy
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)

**Resultado e interpretación.** La celda confirma el entorno utilizado. El laboratorio está diseñado para CPU y se ejecuta en pocos minutos; una GPU disponible se utiliza sin ser requisito.

### Generar interacciones implícitas
Una interacción ocurre con mayor probabilidad cuando los gustos del cliente se alinean con el perfil del producto o cuando el producto es popular. Conservamos clientes con al menos cuatro interacciones.

In [ ]:
n_users_raw, n_items, n_factors, n_categories = 500, 150, 4, 6
item_category = rng.integers(0, n_categories, n_items)
category_profiles = rng.normal(0, 1, (n_categories, n_factors))
item_factors = category_profiles[item_category] + rng.normal(0, 0.4, (n_items, n_factors))
user_factors = rng.normal(0, 1, (n_users_raw, n_factors))
item_popularity = rng.normal(0, 1, n_items)

affinity = 1.5 * user_factors @ item_factors.T / np.sqrt(n_factors) + 0.8 * item_popularity[None, :]
probability = 1 / (1 + np.exp(-(affinity - 4.0)))
interactions = rng.random((n_users_raw, n_items)) < probability
keep = interactions.sum(axis=1) >= 4
interactions, user_factors = interactions[keep], user_factors[keep]
n_users = len(interactions)
counts = interactions.sum(axis=1)
print('clientes:', n_users, '| productos:', n_items)
print('densidad:', round(interactions.mean(), 4), '| interacciones por cliente (mediana):', int(np.median(counts)))

**Resultado e interpretación.** La matriz es dispersa: cada cliente interactúa con una fracción pequeña del catálogo. Esa dispersión es la razón de usar embeddings de baja dimensión, que comparten información entre clientes y productos parecidos.

### Partición *leave-one-out*
Por cliente, reservamos una interacción al azar para validación y otra para test; el resto es train. Así cada cliente tiene exactamente un producto objetivo por evaluación.

In [ ]:
train_matrix = interactions.copy()
val_item = np.empty(n_users, dtype=int)
test_item = np.empty(n_users, dtype=int)
for user in range(n_users):
    held_out = rng.choice(np.flatnonzero(interactions[user]), size=2, replace=False)
    val_item[user], test_item[user] = held_out
    train_matrix[user, held_out] = False

train_pairs = np.argwhere(train_matrix)  # columnas: usuario, producto
print('pares de train:', len(train_pairs), '| validación y test: un producto por cliente')

**Resultado e interpretación.** Ningún producto de validación o test aparece en train para su cliente. Al evaluar excluiremos del ranking los productos ya vistos en train, porque recomendar algo que el cliente ya compró no mide la capacidad de descubrir.

### Métricas de ranking y baseline de popularidad
Para cada cliente ordenamos todos los productos no vistos. HitRate@10 indica si el producto objetivo quedó entre los 10 primeros; NDCG@10 premia además que esté más arriba: $1/\log_2(\text{posición}+1)$. La cobertura es la fracción del catálogo que aparece en alguna lista de top 10.

In [ ]:
def evaluate_ranking(scores, target_items, exclude, k=10):
    scores = scores.astype(float).copy()
    scores[exclude] = -np.inf
    target_scores = scores[np.arange(n_users), target_items]
    rank = (scores > target_scores[:, None]).sum(axis=1)  # 0 = primera posición
    hit = rank < k
    ndcg = np.where(hit, 1 / np.log2(rank + 2), 0.0)
    top_k = np.argsort(-scores, axis=1)[:, :k]
    return {'hit_rate@10': round(float(hit.mean()), 4), 'ndcg@10': round(float(ndcg.mean()), 4),
            'cobertura': round(len(np.unique(top_k)) / n_items, 4)}


exclude_val = train_matrix
exclude_test = train_matrix.copy()
exclude_test[np.arange(n_users), val_item] = True  # en test, el producto de validación ya es conocido

popularity_scores = np.tile(train_matrix.sum(axis=0), (n_users, 1))
random_scores = rng.random((n_users, n_items))
print('aleatorio   validación:', evaluate_ranking(random_scores, val_item, exclude_val))
print('popularidad validación:', evaluate_ranking(popularity_scores, val_item, exclude_val))

**Resultado e interpretación.** El azar acierta aproximadamente 10 de cada ~140 productos candidatos. La popularidad es una referencia exigente cuando la demanda está concentrada, pero recomienda casi la misma lista a todos: observa su cobertura.

### Factorización matricial y NCF
Ambos modelos tienen embeddings de usuario y producto. `score_all` calcula la afinidad de cada cliente con todo el catálogo para evaluar el ranking completo.

In [ ]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, dim):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        for embedding in (self.user_embedding, self.item_embedding):
            nn.init.normal_(embedding.weight, std=0.1)
        for bias in (self.user_bias, self.item_bias):
            nn.init.zeros_(bias.weight)

    def forward(self, users, items):
        dot = (self.user_embedding(users) * self.item_embedding(items)).sum(dim=1)
        return dot + self.user_bias(users).squeeze(1) + self.item_bias(items).squeeze(1)

    def score_all(self, users):
        return (self.user_embedding(users) @ self.item_embedding.weight.T
                + self.user_bias(users) + self.item_bias.weight.T)


class NeuralCF(nn.Module):
    def __init__(self, n_users, n_items, dim, hidden=32):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.mlp = nn.Sequential(nn.Linear(2 * dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2),
                                 nn.ReLU(), nn.Linear(hidden // 2, 1))

    def forward(self, users, items):
        pair = torch.cat([self.user_embedding(users), self.item_embedding(items)], dim=1)
        return self.mlp(pair).squeeze(1)

    def score_all(self, users):
        n_u = len(users)
        all_items = torch.arange(self.item_embedding.num_embeddings, device=users.device)
        return self.forward(users.repeat_interleave(len(all_items)), all_items.repeat(n_u)).view(n_u, -1)


for model in (MatrixFactorization(n_users, n_items, 8), NeuralCF(n_users, n_items, 8)):
    print(type(model).__name__, '| parámetros:', sum(p.numel() for p in model.parameters()))

**Resultado e interpretación.** MF solo puede expresar afinidades como productos punto; NCF puede aprender combinaciones no lineales, pero tiene más parámetros y nada garantiza que los aproveche con datos escasos.

### Entrenar con muestreo negativo
En cada época, por cada par positivo de train muestreamos `n_negatives` productos que el cliente no tiene en train. Usamos *early stopping* según NDCG@10 de validación.

In [ ]:
def sample_epoch(n_negatives):
    pos_users, pos_items = train_pairs[:, 0], train_pairs[:, 1]
    neg_users = np.repeat(pos_users, n_negatives)
    neg_items = rng.integers(0, n_items, len(neg_users))
    clash = train_matrix[neg_users, neg_items]
    while clash.any():  # vuelve a muestrear los "negativos" que en realidad son positivos de train
        neg_items[clash] = rng.integers(0, n_items, clash.sum())
        clash = train_matrix[neg_users, neg_items]
    users = np.concatenate([pos_users, neg_users])
    items = np.concatenate([pos_items, neg_items])
    labels = np.concatenate([np.ones(len(pos_users)), np.zeros(len(neg_users))])
    return (torch.tensor(users, dtype=torch.long), torch.tensor(items, dtype=torch.long),
            torch.tensor(labels, dtype=torch.float32))


def model_scores(model):
    model.eval()
    with torch.no_grad():
        return model.score_all(torch.arange(n_users, device=device)).cpu().numpy()


def train_recommender(build_model, epochs=30, lr=0.01, weight_decay=1e-5, n_negatives=4, batch_size=1024, patience=5):
    torch.manual_seed(SEED)
    model = build_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history, best_ndcg, best_state, wait = [], -1.0, None, 0
    for epoch in range(epochs):
        model.train()
        loader = DataLoader(TensorDataset(*sample_epoch(n_negatives)), batch_size=batch_size, shuffle=True,
                            generator=torch.Generator().manual_seed(SEED + epoch))
        total, n_rows = 0.0, 0
        for users, items, labels in loader:
            users, items, labels = users.to(device), items.to(device), labels.to(device)
            loss = F.binary_cross_entropy_with_logits(model(users, items), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total, n_rows = total + loss.item() * len(labels), n_rows + len(labels)
        metrics = evaluate_ranking(model_scores(model), val_item, exclude_val)
        history.append({'epoch': epoch, 'train_loss': total / n_rows, **metrics})
        if metrics['ndcg@10'] > best_ndcg + 1e-4:
            best_ndcg, best_state, wait = metrics['ndcg@10'], copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, history


recommenders, histories = {}, {}
for name, build_model in [('mf_d8', lambda: MatrixFactorization(n_users, n_items, 8)),
                          ('ncf_d8', lambda: NeuralCF(n_users, n_items, 8))]:
    start = time.perf_counter()
    recommenders[name], histories[name] = train_recommender(build_model)
    print(f'{name}: {len(histories[name])} épocas | {time.perf_counter() - start:.1f} s')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for name, history in histories.items():
    axes[0].plot([h['train_loss'] for h in history], label=name)
    axes[1].plot([h['ndcg@10'] for h in history], label=name)
axes[0].set_title('Pérdida train (con negativos)')
axes[1].set_title('NDCG@10 validación')
for axis in axes:
    axis.set_xlabel('época')
axes[1].legend()
plt.tight_layout()
plt.show()

**Resultado e interpretación.** La pérdida de train baja de forma sostenida, pero el criterio de selección es el ranking de validación. Si NDCG@10 se estanca mientras la pérdida sigue bajando, el modelo está aprendiendo a separar negativos aleatorios fáciles, no a ordenar mejor los productos relevantes.

### Comparar con la popularidad y evaluar en test
Comparamos en validación, elegimos por NDCG@10 y evaluamos una vez en test. Incluimos la cobertura de catálogo.

In [ ]:
candidates = {'popularidad': popularity_scores, **{name: model_scores(m) for name, m in recommenders.items()}}
validation = {name: evaluate_ranking(scores, val_item, exclude_val) for name, scores in candidates.items()}
for name, metrics in validation.items():
    print(f'{name:12s} validación: {metrics}')

chosen = max(validation, key=lambda name: validation[name]['ndcg@10'])
print(f'\nElegido por validación: {chosen}')
print('test:', evaluate_ranking(candidates[chosen], test_item, exclude_test))
print('popularidad en test (referencia):', evaluate_ranking(popularity_scores, test_item, exclude_test))

**Resultado e interpretación.** Compara aciertos y cobertura a la vez. Un modelo personalizado que iguala a la popularidad en HitRate pero duplica la cobertura puede ser preferible para el negocio; uno que la supera ampliamente confirma que los gustos latentes son aprendibles a partir del comportamiento.

### Productos similares a partir de embeddings
Los embeddings de producto de MF permiten responder "quienes compraron esto también podrían querer...". Medimos qué fracción de los 5 vecinos más cercanos (similitud coseno) comparte categoría, frente a lo esperado al azar.

In [ ]:
item_vectors = recommenders['mf_d8'].item_embedding.weight.detach().cpu().numpy()
normalized = item_vectors / np.linalg.norm(item_vectors, axis=1, keepdims=True)
similarity = normalized @ normalized.T
np.fill_diagonal(similarity, -np.inf)
neighbors = np.argsort(-similarity, axis=1)[:, :5]
same_category = (item_category[neighbors] == item_category[:, None]).mean()
expected = np.mean([(item_category == c).sum() - 1 for c in item_category]) / (n_items - 1)
print('fracción de vecinos de la misma categoría:', round(float(same_category), 3),
      '| esperado al azar:', round(float(expected), 3))

user = int(np.argmax(train_matrix.sum(axis=1)))
user_scores = candidates['mf_d8'][user].copy()
user_scores[train_matrix[user]] = -np.inf
print(f'\nCliente {user}: categorías en su historial ->',
      np.bincount(item_category[train_matrix[user]], minlength=n_categories))
print('categorías de su top 10 recomendado ->',
      np.bincount(item_category[np.argsort(-user_scores)[:10]], minlength=n_categories))

**Resultado e interpretación.** La categoría nunca se entregó al modelo, igual que el riesgo de sucursal en el laboratorio 20. Si los vecinos comparten categoría muy por encima del azar, los embeddings recuperaron la estructura del catálogo solo a partir del comportamiento. La comparación de categorías del cliente sirve como explicación simple de una recomendación.

## Limitaciones y conclusiones clave
Los datos son sintéticos y la evaluación *leave-one-out* con un solo producto objetivo es ruidosa. En sistemas reales la partición debe respetar el tiempo (entrenar con el pasado y evaluar con interacciones futuras), y los usuarios o productos nuevos (*cold start*) requieren atributos adicionales. Los negativos muestreados pueden ser positivos no observados.

- MF y NCF aprenden embeddings de usuarios y productos a partir de interacciones implícitas y muestreo negativo.
- Un recomendador se evalúa con métricas de ranking y se compara contra la popularidad.
- La cobertura y la explicabilidad son parte de la evaluación, no un complemento.

## Ejercicios progresivos
1. **Guiado:** cambia `n_negatives` a 1 y a 10 para MF y compara NDCG@10 de validación y tiempo de entrenamiento.
2. **Independiente:** entrena MF con dimensiones 2, 4, 16 y 32; grafica NDCG@10 de validación frente a la dimensión y relaciónalo con los 4 factores latentes del generador.
3. **Desafío:** simula un cliente nuevo con solo dos interacciones. Congela los embeddings de producto, optimiza solo un vector de usuario nuevo (*fold-in*) con esas dos interacciones y compara su top 10 con el de popularidad.

## Conexión con el siguiente laboratorio
Ventas y comportamiento también tienen dimensión temporal. En el laboratorio 22 comparamos MLP, CNN 1D y LSTM para pronosticar demanda.